# At-Risk Learner Prediction Model

Train a GradientBoostingClassifier on synthetic student data to identify learners at risk of underperformance.

**Job Responsibility:** Predictive modeling for learner performance prediction, early intervention identification, and resource optimization.

**Key Deliverables:**
- Synthetic dataset (500 students, seed=42)
- Feature engineering (engagement_score, quiz_trend, attendance_x_quiz)
- 5-fold CV with MLflow tracking
- SHAP explanations for model interpretability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")

## 1. Generate Synthetic Student Data

Since we can't use real FERPA-protected student data, we generate realistic synthetic data representing 500 medical students.

In [ ]:
np.random.seed(42)
n = 500

data = pd.DataFrame({
    'student_id': range(n),
    'quiz_avg': np.random.normal(75, 12, n),
    'attendance_pct': np.random.beta(8, 2, n) * 100,
    'resource_views': np.random.poisson(15, n),
    'forum_posts': np.random.poisson(3, n),
    'time_on_platform_hrs': np.random.gamma(5, 2, n),
    'prior_gpa': np.random.normal(3.2, 0.5, n).clip(0, 4),
})

# At-risk label: students with low quiz scores OR low attendance
data['at_risk'] = ((data['quiz_avg'] < 65) | (data['attendance_pct'] < 70)).astype(int)

print(f"Dataset shape: {data.shape}")
print(f"At-risk rate: {data['at_risk'].mean():.1%}")
data.head()

## 2. Feature Engineering

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.prediction.at_risk_model import AtRiskPipeline

pipeline = AtRiskPipeline()
features_df = pipeline.create_features(data)

print("Engineered features:")
print(features_df[['engagement_score', 'quiz_trend', 'attendance_x_quiz']].describe())

## 3. Train Model with MLflow Tracking

In [ ]:
feature_cols = [
    'quiz_avg', 'attendance_pct', 'resource_views', 'forum_posts',
    'time_on_platform_hrs', 'prior_gpa',
    'engagement_score', 'quiz_trend', 'attendance_x_quiz',
]

X = features_df[feature_cols]
y = features_df['at_risk']

pipeline.train_and_log(X, y, experiment_name='at_risk_prediction')
print("Training complete. Check MLflow UI for metrics.")

## 4. SHAP Explanations

SHAP values let advisors understand *why* a student is flagged -- is it attendance? quiz trends? -- so they can target the right intervention.

In [ ]:
import shap

explainer = shap.TreeExplainer(pipeline.pipeline.named_steps['model'])

# Scale features the same way the pipeline does
X_scaled = pipeline.pipeline.named_steps['scaler'].transform(X)
shap_values = explainer.shap_values(X_scaled)

# Summary plot
shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'exports' / 'shap_at_risk.png'), dpi=150)
plt.show()
print("SHAP summary plot saved.")

## 5. Sample Predictions

Show predictions for a few sample students to demonstrate model output.

In [ ]:
probs = pipeline.predict(X)

results = features_df[['student_id', 'quiz_avg', 'attendance_pct', 'at_risk']].copy()
results['risk_probability'] = probs

print("\nHighest risk students:")
print(results.nlargest(10, 'risk_probability').to_string(index=False))

print(f"\nTotal at-risk (predicted > 0.5): {(probs > 0.5).sum()} / {len(probs)}")